# Assemble the S&P 500 Item 1 Table

| Item | Description |
|---|---|
| Goal | Turn the 21,867 extracted Item 1 files and the 432 gaps the API left into one table: one row per S&P 500 annual report with the Item 1 (Business) text, where it came from, and whether it is usable. Three things happen in order, each measured: fill the gaps from the archived filings, clean the text for use as model input, and cut the text of the next item where the API ran past the boundary. |
| Input | From D1.3.3, run `sp500_item1`: `Interim/sp500_item1/item1/<cik>/<accessionNo>.txt` and `Interim/sp500_item1/status/extract_*.jsonl`. `Processed/filings.parquet` from D1.3.2. `Raw/form10k/*.zip` from D1.3.1, read by the local extractor for the 432 gaps. |
| Output | `Processed/sp500_10k_item1.parquet` in the D1.3 scratch folder: 22,299 rows, about 370 MB, with the firm identifiers, the provenance columns, and `item1`, the cleaned text. |
| Variables | • `source`: `sec-api`, `local`, or `none`, where the text came from<br>• `recovery_note`: the local extractor's status for a gap<br>• `cut_kind`, `cut_chars`: `leading_toc`, `next_section`, or `none`, and how many characters were removed<br>• `chars`: length of `item1`<br>• `starts_at_item1`: the text opens with an Item 1 heading<br>• `incorporated_by_reference`: under 1,000 characters, a pointer to another document rather than a description<br>• `usable`: at least 1,000 characters |
| Run | Last run 2026-08-20; the cell outputs shown are from that run. Not rerun: T3 was built on this table. It now sits in the Processed folder named above. |

## Step 1. Import packages

In [ ]:
import html
import json
import re
import zipfile
from pathlib import Path

import pandas as pd

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D1 - Data Collection and Processing/D1.3 - SEC Form 10-K Corporate Filings")
NOTEBOOK_NAME = "D1.3.4 - SEC 10-K - Assemble SP500 Item 1 Panel"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
CONTAINER_FOLDER = RAW_FOLDER / "form10k"  # the archived filings, written by D1.3.1
RUN_FOLDER = INTERIM_FOLDER / "sp500_item1"  # the Item 1 extraction run of D1.3.3

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Set the usability threshold

Below 1,000 characters a section is a pointer, not a business description. 216 API results fall under it, and every one inspected was an incorporation by reference.

In [1]:
MIN_USABLE_CHARS = 1000

print(f"run    : {RUN_FOLDER}")
print(f"output : {PROCESSED_FOLDER}")

run    : /scratch2/acroh/projects/llm_method/data/interim/sec/sp500_item1
output : /project2/njia_1089/acroh/projects/llm_method/data/raw/03_sp500_10k_item1


## Step 4. Load the status log and the filing index

The status log says what was attempted. The filing index supplies GVKEY and the fiscal year, so the table carries them without a later lookup.

In [2]:
rows = [json.loads(line) for p in sorted((RUN_FOLDER / "status").glob("extract_*.jsonl")) for line in p.open() if line.strip()]
status = pd.DataFrame(rows).drop_duplicates(["accessionNo", "item"], keep="last")

idx = pd.read_parquet(PROCESSED_FOLDER / "filings.parquet")
sp = idx.loc[idx["in_sp500"]].copy()

assert len(status) == len(sp), f"{len(status):,} status rows against {len(sp):,} S&P 500 filings: the extraction did not cover the index"

print(f"status rows : {len(status):,}")
print(status["status"].value_counts().to_string())
print(f"\nS&P 500 filings in the index: {len(sp):,}")

status rows : 22,299
status
ok       21867
empty      432

S&P 500 filings in the index: 22,299


## Step 5. Define how an archived filing is read

The Extractor returned nothing for 432 filings, 1.94 percent, and the failure is deterministic. FactSet, Marathon Oil, and Iron Mountain all have full Business sections it declines to return, so the archived filing is read instead. Old `.txt` archives hold the whole SGML submission, so the first document whose type starts with 10-K is taken. HTML bodies wrap source lines arbitrarily, so their newlines are removed before tag breaks are inserted.

In [ ]:
_DOC_RE = re.compile(r"(?is)<DOCUMENT>(.*?)(?:</DOCUMENT>|\Z)")
_TYPE_RE = re.compile(r"(?i)<TYPE>\s*([^\s<]+)")
_TEXT_RE = re.compile(r"(?is)<TEXT>(.*?)(?:</TEXT>|\Z)")

_RE_COMMENT = re.compile(r"(?is)<!--.*?-->")
_RE_SCRIPT = re.compile(r"(?is)<(script|style)\b[^>]*>.*?</\1\s*>")
_RE_IXHIDE = re.compile(r"(?is)<(ix:header|ix:hidden)\b[^>]*>.*?</\1\s*>")
_RE_XMLDECL = re.compile(r"(?is)<\?xml.*?\?>")
_RE_BLOCK = re.compile(  # block elements become a newline
    r"(?is)<\s*/?\s*(br|p|div|tr|table|thead|tbody|h[1-6]|li|ul|ol|center|hr|"
    r"caption|blockquote|dt|dd|article|section|body)\b[^>]*>"
)
_RE_CELL = re.compile(r"(?is)<\s*/?\s*(td|th)\b[^>]*>")  # table cells become a tab, keeping "Item 1." and "Business" on one line
_RE_TAG = re.compile(r"(?s)<[^>\n]{0,2000}>")
_RE_PAGE = re.compile(r"(?i)<PAGE[^>]*>")
_RE_UNISPACE = re.compile("[   -   　]")  # thin, hair, and figure spaces between "Item" and "1"
_RE_ZEROWIDTH = re.compile("[​‌‍⁠﻿­]")
_RE_HTMLISH = re.compile(r"(?i)<\s*(!doctype\s+html|html\b|body\b|/?div\b|/?font\b|/?span\b|/?td\b|br\s*/?>)")
_RE_CLOSERS = re.compile(r"(?i)</\s*(td|tr|div|font|span|p)\s*>")


def decode_bytes(b):
    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return b.decode(encoding)
        except UnicodeDecodeError:
            continue
    return b.decode("latin-1", "replace")


def select_primary_document(raw):
    """The <TEXT> body of the first <DOCUMENT> whose <TYPE> starts with 10-K; HTML files are returned unchanged."""
    if "<DOCUMENT>" not in raw[:400000]:
        return raw
    best = None
    for d in _DOC_RE.finditer(raw):
        body = d.group(1)
        m = _TYPE_RE.search(body[:2000])
        t = (m.group(1) if m else "").upper()
        if t.startswith("10-K") or t.startswith("10K"):
            tm = _TEXT_RE.search(body)
            return tm.group(1) if tm else body
        if best is None:
            tm = _TEXT_RE.search(body)
            best = tm.group(1) if tm else body
    return best if best is not None else raw


def is_html(doc):
    """Old EDGAR .txt bodies are fixed-width ASCII where newlines carry meaning; HTML bodies are not."""
    if _RE_HTMLISH.search(doc[:300000]):
        return True
    return len(_RE_CLOSERS.findall(doc[:500000])) >= 5


def markup_to_text(s):
    if is_html(s):
        s = s.replace("\r", " ").replace("\n", " ")
    s = _RE_XMLDECL.sub(" ", s)
    s = _RE_COMMENT.sub(" ", s)
    s = _RE_SCRIPT.sub(" ", s)
    s = _RE_IXHIDE.sub(" ", s)
    s = _RE_PAGE.sub("\n", s)
    s = _RE_BLOCK.sub("\n", s)
    s = _RE_CELL.sub("\t", s)
    s = _RE_TAG.sub("", s)
    s = html.unescape(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _RE_UNISPACE.sub(" ", s)
    s = _RE_ZEROWIDTH.sub("", s)
    s = s.replace(" ", "\n").replace(" ", "\n")
    s = re.sub(r"[ \t\f\v]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


_ZIP_CACHE = {}


def _open_zip(container):
    """One open handle per container, with an index from accession folder to its members."""
    path = str(CONTAINER_FOLDER / container)
    z = _ZIP_CACHE.get(path)
    if z is None:
        z = zipfile.ZipFile(path)
        _ZIP_CACHE[path] = z
        z._member_index = {}
        for n in z.namelist():
            parts = n.split("/")
            if len(parts) >= 2:
                z._member_index.setdefault(parts[-2], []).append(n)
    return z


def read_archived_document(accession_no, container):
    """The primary document of one filing out of its container, as text."""
    z = _open_zip(container)
    members = z._member_index.get(accession_no.replace("-", ""), [])
    docs = [m for m in members if not m.endswith("metadata.json")]
    if not docs:
        return ""
    docs.sort(key=lambda m: (0 if m.lower().endswith((".htm", ".html")) else 1, len(m)))
    return decode_bytes(z.read(docs[0]))

## Step 6. Define how item headings are found

An item heading such as `Item 1.`, `ITEM 1:`, or `Items 1 and 2.` is found at the start of a line. Filers also write `Item l.` with a letter or `ITEM I.` with a roman numeral, or omit the word Item, so those forms are added. A heading is told from a cross-reference by what follows it on the line: a title or nothing, never a lower-case continuation. A row with dot leaders or a trailing page number is a table of contents.

In [ ]:
_ITEM_RE = re.compile(
    r"""(?imx)
    ^[ \t]*
    (?:PART\s+[IVX]+[ \t]*[.,:\-]?[ \t]*)?
    I[ \t]?T[ \t]?E[ \t]?M[ \t]?S?
    [ \t]*[\.\:\-‐-―]?[ \t]*
    (\d{1,2})[ \t]*([A-Ca-c])?
    (?![\w])
    (?:
        [ \t]*[\.\:\-‐-―]?[ \t]*
        (?:AND|&|,|THROUGH|TO)
        [ \t]*(?:ITEMS?[ \t]*)?
        (\d{1,2})[ \t]*([A-Ca-c])?
        (?![\w])
    )?
    """
)
_ITEM_TITLES = ("business", "properties", "propertie", "legal proceedings", "risk factors",
                "unresolved staff comments", "mine safety disclosures", "submission of matters to a vote of security holders")
# "1. Business." used as a heading with the word Item omitted, anchored to the end of the line so that a note title
# such as "1. Business and Summary of Significant Accounting Policies" cannot match.
_BARE_ITEM_RE = re.compile(r"(?im)^[ \t]*(\d{1,2})[ \t]*([A-Ca-c])?[\.\:][ \t]*(" + "|".join(t.replace(" ", r"\s+") for t in _ITEM_TITLES) + r")s?[ \t]*\.?[ \t]*$")
_ITEM1_LENIENT = re.compile(r"(?im)^[ \t]*(?:PART\s+[IVX]+[ \t]*[.,:\-]?[ \t]*)?ITEMS?[ \t]*[\.\:\-]?[ \t]*(?:1|I|l)[ \t]*[\.\:\-]?[ \t]*(?:BUSINESS|DESCRIPTION OF (?:THE )?BUSINESS)")

_RANK = {}
for _n in range(0, 20):
    _RANK[(str(_n), "")] = _n * 10
    for _i, _s in enumerate("abc", start=1):
        _RANK[(str(_n), _s)] = _n * 10 + _i


def _rank(num, suf):
    return _RANK.get((num, (suf or "").lower()), int(num) * 10)


TITLE_WORDS = ("business", "propert", "risk factor", "unresolved", "legal proceeding", "mine safety", "submission of matters",
               "market for", "selected financial", "management's discussion", "quantitative and qualitative", "financial statements",
               "changes in and disagreements", "controls and procedures", "description of business", "general", "executive officers")


class Marker:
    __slots__ = ("pos", "line_start", "line_end", "line", "num", "suf", "num2", "suf2", "rank_lo", "rank_hi", "text")

    def __repr__(self):
        return f"<Marker {self.text!r} @{self.pos}>"


def _mk(text, pos, num, suf, matched):
    mk = Marker()
    ls = text.rfind("\n", 0, pos) + 1
    le = text.find("\n", pos)
    le = len(text) if le < 0 else le
    mk.pos, mk.line_start, mk.line_end = pos, ls, le
    mk.line = text[ls:le]
    mk.num, mk.suf, mk.num2, mk.suf2 = num, suf, None, ""
    mk.rank_lo = mk.rank_hi = _rank(num, suf)
    mk.text = matched.strip()
    return mk


def find_markers(text):
    """Every strict item marker in the document, in order."""
    out = []
    for m in _ITEM_RE.finditer(text):
        mk = Marker()
        mk.pos = m.start()
        ls = text.rfind("\n", 0, m.start()) + 1
        le = text.find("\n", m.start())
        le = len(text) if le < 0 else le
        mk.line_start, mk.line_end = ls, le
        mk.line = text[ls:le]
        mk.num, mk.suf = m.group(1), (m.group(2) or "")
        mk.num2, mk.suf2 = m.group(3), (m.group(4) or "")
        mk.rank_lo = _rank(mk.num, mk.suf)
        mk.rank_hi = _rank(mk.num2, mk.suf2) if mk.num2 else mk.rank_lo
        if mk.rank_hi < mk.rank_lo:
            mk.rank_hi = mk.rank_lo
        mk.text = m.group(0).strip()
        out.append(mk)
    return out


def augment_lenient(text, markers):
    """Add the letter-l, roman-numeral, and no-word-Item forms as candidates where the strict regex did not fire."""
    have = {m.line_start for m in markers}
    extra = []
    for m in _ITEM1_LENIENT.finditer(text):
        ls = text.rfind("\n", 0, m.start()) + 1
        if ls in have:
            continue
        extra.append(_mk(text, m.start(), "1", "", m.group(0)))
        have.add(ls)
    for m in _BARE_ITEM_RE.finditer(text):
        ls = text.rfind("\n", 0, m.start()) + 1
        if ls in have:
            continue
        extra.append(_mk(text, m.start(), m.group(1), (m.group(2) or ""), m.group(0)))
        have.add(ls)
    if not extra:
        return markers
    return sorted(markers + extra, key=lambda x: x.pos)


_SENTENCE_LEAD = re.compile(r"^[a-z]")


def _looks_like_heading(text, mk):
    """A short line whose remainder after the item number is empty, a capital, or a known title; never a lower-case continuation."""
    line = mk.line.strip()
    if len(line) > 160:
        return False
    rest = text[mk.pos + len(mk.text): mk.line_end].strip(" \t.:;,-–—()‐‑‒–—")
    if rest == "":
        nxt = text[mk.line_end: mk.line_end + 120].strip()
        nxt = nxt.split("\n")[0].strip() if nxt else ""
        rest = nxt
        if rest == "":
            return True
    low = rest.lower()
    if any(low.startswith(w) for w in TITLE_WORDS):
        return len(line) <= 160
    if _SENTENCE_LEAD.match(rest):
        return False
    return len(rest) <= 70


def _toc_line(text, mk):
    """Dot leaders, or a trailing page number, on the heading's own line."""
    line = mk.line
    if re.search(r"\.{4,}", line):
        return True
    if re.search(r"[\s\t]\d{1,3}\s*$", line) and len(line) <= 120:
        return True
    return False

## Step 7. Define how Item 1 is chosen

A table of contents is a dense, strictly ascending run of at least four item markers, and candidates inside one are set aside. Item 1 ends at the next adjacent heading, 1A or 2, and only failing that at any later heading. The earliest candidate at least half as long as the longest wins; later ones are cross-references from the MD&A. Against the API on 5,314 filings the result agrees at a median token Jaccard of 0.9976, and its known failure is running past the end of Item 1.

In [ ]:
_REF_RE = re.compile(r"(?is)(incorporated\s+(?:herein\s+)?by\s+reference|is\s+incorporated\s+herein|see\s+pages?\s+\d|"
                     r"pages?\s+\d[\d\s,\-and]*of\s+(?:the\s+)?(?:registrant|company)?)")
_XREF_RE = re.compile(r"(?is)(cross[\s\-]?reference|incorporated\s+(?:herein\s+)?by\s+reference|annual\s+report\s+to\s+(?:share|stock)holders)")
MIN_SECTION_CHARS = 1000


def _toc_blocks(markers, text):
    """Spans of the text that are a table of contents: a strictly ascending run of at least four markers with small gaps."""
    blocks = []
    i = 0
    n = len(markers)
    while i < n:
        j = i
        seen = {markers[i].rank_lo}
        while (j + 1 < n and (markers[j + 1].pos - markers[j].line_end) < 400 and markers[j + 1].rank_lo > markers[j].rank_lo):
            j += 1
            seen.add(markers[j].rank_lo)
        if len(seen) >= 4 and j > i:
            blocks.append((markers[i].line_start, markers[j].line_end))
        i = j + 1
    return blocks


def _section_for(text, markers, k, pos=None, covered=None):
    """Start and end of the section opened by marker k: first a near successor (within two ranks), then any later heading."""
    if pos is None:
        start = markers[k]
        covered = start.rank_hi
        pos = start.line_start
    body_start = pos
    for window in (20, 10000):
        for j in range(len(markers)):
            e = markers[j]
            if e.line_start <= body_start:
                continue
            if e.rank_lo <= covered or e.rank_lo > covered + window:
                continue
            if not _looks_like_heading(text, e):
                continue
            return body_start, e.line_start, e
    return body_start, len(text), None


def _row_title(text, mk):
    """The title that goes with a marker: the rest of its line, else the next non-empty line."""
    rest = text[mk.pos + len(mk.text): mk.line_end].strip(" \t.:;,-‐-―()")
    if not rest:
        nxt = text[mk.line_end: mk.line_end + 200].strip("\n")
        rest = nxt.split("\n")[0].strip() if nxt else ""
    rest = re.sub(r"[\s.]*\d[\d\s,\-–]*$", "", rest).strip(" \t.:;,-")
    return rest


def _standalone_line(text, title, after):
    if not title or len(title) < 4 or len(title) > 60:
        return None
    rx = re.compile(r"(?im)^[ \t]*" + re.escape(title) + r"[ \t]*\.?[ \t]*$")
    m = rx.search(text, after)
    return m.start() if m else None


def _toc_title_fallback(text, markers, best):
    """Integrated annual reports carry item numbers only in a cross-reference table; follow the row's title into the body."""
    mk, endmk = best["marker"], best["endmk"]
    t1 = _row_title(text, mk)
    if not t1 or "business" not in t1.lower():
        return None
    after = mk.line_end + 200
    s1 = _standalone_line(text, t1, after)
    while s1 is not None and endmk is not None and s1 < endmk.line_end:
        s1 = _standalone_line(text, t1, s1 + 1)
    if s1 is None:
        return None
    t2 = _row_title(text, endmk) if endmk is not None else ""
    e1 = _standalone_line(text, t2, s1 + 1000) if t2 else None
    if e1 is None:
        nxt = [m for m in markers if m.line_start > s1 + 1000 and m.rank_lo > mk.rank_hi and _looks_like_heading(text, m)]
        e1 = nxt[0].line_start if nxt else None
    if e1 is None:
        return None
    body = text[s1:e1].strip()
    if len(body) < MIN_SECTION_CHARS or len(body) > 0.6 * len(text):
        return None
    return {"start": s1, "end": e1, "body": body, "title": t1, "end_title": t2}


def choose_item1(text, markers):
    cands = [k for k, m in enumerate(markers) if m.num == "1" and m.suf == ""]
    if not cands:
        return {"status": "not_found", "text": "", "n_candidates": 0, "note": "no 'Item 1' marker in document"}

    toc = _toc_blocks(markers, text)

    # Some archived "10-K" submissions hold a 10-Q, whose Item 1 is Financial Statements, not Business.
    def _is_10q_item1(mk):
        tail = text[mk.pos: mk.line_end + 40].lower()
        return "financial statements" in tail[:80] and "business" not in tail[:80]

    q_dropped = [k for k in cands if _is_10q_item1(markers[k])]
    cands = [k for k in cands if k not in set(q_dropped)]
    if not cands:
        return {"status": "not_found", "text": "", "n_candidates": 0,
                "note": "only 10-Q style 'Item 1. Financial Statements' headings found - archived document is not a 10-K body"}

    scored = []
    for k in cands:
        mk = markers[k]
        s, e, endmk = _section_for(text, markers, k)
        body = text[s:e].strip()
        in_toc = any(a <= mk.pos <= b for a, b in toc) or _toc_line(text, mk)
        scored.append({"k": k, "start": s, "end": e, "body": body, "len": len(body), "in_toc": in_toc, "marker": mk, "endmk": endmk})

    # Pools from strictest to loosest; within a pool the earliest candidate at least half as long as the longest wins.
    pools = [
        [c for c in scored if not c["in_toc"] and _looks_like_heading(text, c["marker"])],
        [c for c in scored if not c["in_toc"]],
        [c for c in scored if _looks_like_heading(text, c["marker"])],
        scored,
    ]
    best = None
    for pool in pools:
        if not pool:
            continue
        lmax = max(c["len"] for c in pool)
        cand = min([c for c in pool if c["len"] >= 0.5 * lmax], key=lambda c: c["marker"].pos)
        if best is None or cand["len"] > best["len"]:
            if best is None or best["len"] < MIN_SECTION_CHARS:
                best = cand
        if best["len"] >= MIN_SECTION_CHARS:
            break

    body = best["body"]
    note = ""
    status = "ok"
    if len(body) < MIN_SECTION_CHARS:
        alt = _toc_title_fallback(text, markers, best)
        if alt is not None:
            return {"status": "ok", "text": alt["body"], "n_candidates": len(cands),
                    "start_heading": alt["title"][:120], "end_heading": (alt["end_title"] or "<next item heading>")[:120],
                    "combined_items": "1", "note": "cross-reference-table 10-K: located section by its title, not by an item number"}
        window = text[max(0, best["start"] - 1500): best["start"] + 2500]
        if _REF_RE.search(body[:2500]) or _XREF_RE.search(window):
            status = "incorporated_by_reference"
            note = "short Item 1 that points elsewhere (cross-reference index / incorporation by reference); %d chars" % len(body)
        else:
            status = "not_found"
            note = f"best candidate only {len(body)} chars"
    return {
        "status": status,
        "text": body,
        "n_candidates": len(cands),
        "start_heading": best["marker"].line.strip()[:120],
        "end_heading": (best["endmk"].line.strip()[:120] if best["endmk"] else "<end of document>"),
        "combined_items": (f"{best['marker'].num}{best['marker'].suf}" + (f"-{best['marker'].num2}{best['marker'].suf2}" if best["marker"].num2 else "")),
        "note": note,
    }


def extract_item1_from_text(raw):
    doc = select_primary_document(raw)
    text = markup_to_text(doc)
    markers = augment_lenient(text, find_markers(text))
    res = choose_item1(text, markers)
    res["chars"] = len(res["text"])
    res["doc_chars"] = len(text)
    return res


def extract_item1(accession_no, container):
    """Item 1 of one archived filing: status ok, incorporated_by_reference, not_found, or no_document, with the text."""
    if not container:
        return {"status": "no_document", "text": "", "chars": 0, "note": "accession not in filing index"}
    raw = read_archived_document(accession_no, container)
    if not raw:
        return {"status": "no_document", "text": "", "chars": 0, "note": "no document member in archive"}
    res = extract_item1_from_text(raw)
    res["accessionNo"] = accession_no
    res["container"] = container
    return res

## Step 8. Recover the filings the API left empty

The three outcomes are kept apart. `ok` is recovered text. `incorporated_by_reference` means the filing only points elsewhere, which is not a failure and not usable text. `not_found` means no Item 1 heading could be located.

In [3]:
empty = status.loc[status["status"] == "empty", "accessionNo"].tolist()
container = idx.set_index("accessionNo")["container"]

recovered = {}
counts = {"ok": 0, "incorporated_by_reference": 0, "not_found": 0, "error": 0}
for acc in empty:
    try:
        r = extract_item1(acc, container.get(acc))
    except Exception as exc:  # reported, not raised
        counts["error"] += 1
        recovered[acc] = {"status": "error", "text": "", "note": f"{type(exc).__name__}: {exc}"}
        continue
    counts[r.get("status", "not_found")] = counts.get(r.get("status", "not_found"), 0) + 1
    recovered[acc] = r

print(f"attempted recovery on {len(empty):,} filings the API returned nothing for")
for key, value in counts.items():
    if value:
        print(f"  {key:<26} {value:>5,}")
got = [len(r.get("text") or "") for r in recovered.values() if r.get("status") == "ok"]
if got:
    s = pd.Series(got)
    print(f"\nrecovered text: median {s.median():,.0f} chars, max {s.max():,}")

attempted recovery on 432 filings the API returned nothing for
  ok                           334
  incorporated_by_reference     29
  not_found                     69

recovered text: median 44,856 chars, max 252,461


## Step 9. Assemble one row per filing

`source` records where each row's text came from. A later analysis can then drop the locally extracted rows if it wants API-only provenance.

In [4]:
def read_api_text(path):
    try:
        return Path(path).read_text()
    except (OSError, TypeError):
        return ""


records = []
for row in status.itertuples():
    if row.status == "ok":
        text, source, note = read_api_text(row.path), "sec-api", ""
    elif row.accessionNo in recovered:
        r = recovered[row.accessionNo]
        text = r.get("text") or ""
        source = "local" if r.get("status") == "ok" else "none"
        note = r.get("status", "")
    else:
        text, source, note = "", "none", row.status
    records.append({"accessionNo": row.accessionNo, "text_raw": text, "source": source, "recovery_note": note})

items = pd.DataFrame(records)
keep = ["accessionNo", "cik_unpadded", "gvkey", "conm", "companyName", "formType", "fyear", "periodEnd", "filedAt", "filingUrl", "url_bare"]
items = items.merge(sp[keep], on="accessionNo", how="left", validate="one_to_one")

print(items["source"].value_counts().to_string())
print(f"\n{len(items):,} rows")

source
sec-api    21867
local        334
none          98

22,299 rows


## Step 10. Clean the text

The text is fed to a model, so the target is that nothing foreign sits inside a sentence. Page sentinels such as `##PAGE` and bare page numbers land mid-sentence in a fifth of the files, and 48 percent of the documents are hard-wrapped. Page furniture collapses to a newline first. A newline is then removed when its line does not end a sentence and the next starts in lower case, while the section glyph `§` stays as a legal citation.

In [5]:
TABLE_MARK = re.compile(r"##TABLE_(?:START|END)")
UNI_SPACE = re.compile("[   -   　]")
ZERO_WIDTH = re.compile("[​‌‍⁠﻿­]")
PAGE_MARK = re.compile(r"(?i)[ \t]*(?:##[ \t]*PAGE|<PAGE>)[ \t]*")  # page-break sentinels, the API's form and the SGML's
PAGE_NUM = re.compile(r"\n[ \t]*-{0,2}[ \t]*\d{1,4}[ \t]*-{0,2}[ \t]*(?=\n)")  # a line holding only a page number
LEADER = re.compile(r"[.·]{4,}|_{3,}|-{4,}|={3,}|\*{3,}")  # alignment fill; the surrounding text stays
WRAP = re.compile(r"(?<![.:;!?\)\]\"\'])\n(?=[a-z])")  # a newline inside a sentence


def clean_text(t):
    if not t:
        return ""
    t = html.unescape(html.unescape(t))  # twice: some output is double-encoded
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = ZERO_WIDTH.sub("", t)
    t = UNI_SPACE.sub(" ", t)
    t = TABLE_MARK.sub(" ", t)

    t = PAGE_MARK.sub("\n", t)  # page furniture first, so the unwrap below can rejoin the sentence it split
    t = PAGE_NUM.sub("\n", t)
    t = LEADER.sub(" ", t)

    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"[ \t]*\n[ \t]*", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = WRAP.sub(" ", t)

    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


before = items["text_raw"].str.len()
items["text_clean"] = items["text_raw"].map(clean_text)
after = items["text_clean"].str.len()

alive = before > 0
print(f"cleaning removed {(before - after)[alive].sum() / 1e6:.1f}M characters ({(1 - after[alive].sum() / before[alive].sum()):.1%} of the corpus)")
print(f"  median shrink per document: {((1 - after[alive] / before[alive])).median():.1%}")
for name, pat in [("HTML entities", r"&#\d+;|&[a-z]{2,8};"),
                  ("table markers", r"##TABLE_"),
                  ("page sentinels", r"(?i)##\s*PAGE|<PAGE>"),
                  ("bare page numbers", r"\n[ \t]*\d{1,4}[ \t]*\n"),
                  ("dot leaders", r"\.{4,}"),
                  ("mid-sentence newline", r"(?<![.:;!?\)\]])\n(?=[a-z])"),
                  ("4+ space runs", r"[ \t]{4,}"),
                  ("4+ newline runs", r"\n{4,}")]:
    b = items["text_raw"].str.contains(pat, regex=True, na=False).sum()
    a = items["text_clean"].str.contains(pat, regex=True, na=False).sum()
    print(f"  {name:<16} {b:>7,} -> {a:>6,}")

cleaning removed 74.8M characters (6.6% of the corpus)
  median shrink per document: 4.3%


  HTML entities     16,665 ->      0


  table markers     14,731 ->      0


  page sentinels     4,497 ->      0


  bare page numbers   4,351 ->     17


  dot leaders        1,377 ->      0


  mid-sentence newline  17,660 ->  2,658


  4+ space runs      5,038 ->      0


  4+ newline runs    2,716 ->      0


## Step 11. Find the item headings that start a line

A cross-reference has sentence text to its left on the same line; a heading does not. Both `Item 1A` and `Item 2` or `Item 3` are looked for. A filing that predates Item 1A ends at Properties or Legal Proceedings.

In [6]:
HEADING = re.compile(
    r"(?:^|\n)[ \t]*(?:part\s+i[ \t]*[\-–—]?[ \t]*)?"
    r"item[ \t]*(1A|1B|2|3)[ \t]*[\.\:\-–—]?[ \t]*"
    r"(risk[ \t]*factors?|unresolved|propert|legal)",
    re.I)


def line_initial_headings(t):
    """Every item heading that begins its own line, as (position, item)."""
    out = []
    for mt in HEADING.finditer(t):
        line_start = t.rfind("\n", 0, mt.start() + 1) + 1
        if t[line_start:mt.start()].strip():  # text to the left on the same line: a cross-reference, not a heading
            continue
        out.append((mt.start(), mt.group(1).upper()))
    return out


items["headings"] = items["text_clean"].map(line_initial_headings)
has = items["headings"].map(len) > 0
xref = items["text_clean"].str.contains(r"item\s*1A", case=False, regex=True, na=False) & ~has

print(f"line-initial item headings found in : {int(has.sum()):>6,} documents")
print(f"'Item 1A' present only as prose      : {int(xref.sum()):>6,} documents")
print(f"neither                              : {int((~has & ~xref).sum()):>6,} documents")

line-initial item headings found in :    211 documents
'Item 1A' present only as prose      :  6,301 documents
neither                              : 15,787 documents


## Step 12. Cut the front, or cut the back

A run of headings in the first tenth of the document is a leading table of contents, and everything up to the last of them goes. A heading anywhere else starts the next section, and everything from it goes. There is no minimum on what follows. Both cuts are recorded per row.

In [7]:
TOC_ZONE = 0.10  # a heading this early is part of a front table of contents
TOC_GAP = 400  # consecutive table-of-contents rows sit close together
MIN_TAIL = 2000  # guards the table-of-contents rule only; the tail cut has no minimum


def cut(row):
    t, hs = row["text_clean"], row["headings"]
    if not t or not hs:
        return pd.Series([t, "none", 0])

    n = len(t)
    front = [p for p, _ in hs if p < n * TOC_ZONE]
    # A table of contents is a run of headings; a single early heading is more likely a real short Item 1.
    if len(front) >= 2 and max(b - a for a, b in zip(front, front[1:])) < TOC_GAP:
        end = max(front)
        end = t.find("\n", end + 40)
        if end > 0 and (n - end) > MIN_TAIL:
            return pd.Series([t[end:].strip(), "leading_toc", end])

    tail = [p for p, _ in hs if p >= n * TOC_ZONE]
    if tail:
        return pd.Series([t[:min(tail)].strip(), "next_section", n - min(tail)])
    return pd.Series([t, "none", 0])


items[["item1", "cut_kind", "cut_chars"]] = items.apply(cut, axis=1)

print(items["cut_kind"].value_counts().to_string())
c = items.loc[items["cut_kind"] != "none"]
if len(c):
    print(f"\n{len(c):,} documents cut, {c['cut_chars'].sum() / 1e6:.2f}M characters removed")
    print(c.groupby("cut_kind")["cut_chars"].describe()[["count", "50%", "max"]].to_string())

cut_kind
none            22089
next_section      205
leading_toc         5

210 documents cut, 3.30M characters removed
              count     50%       max
cut_kind                             
leading_toc     5.0   172.0     762.0
next_section  205.0  1399.0  271248.0


## Step 13. Label every row

`usable` is the column an analysis should filter on. The text has to exist and be long enough to be a description rather than a pointer.

In [8]:
items["chars"] = items["item1"].str.len()
items["incorporated_by_reference"] = (items["chars"] > 0) & (items["chars"] < MIN_USABLE_CHARS)
items["starts_at_item1"] = items["item1"].str.match(r"\W{0,80}(part\s+i\b.{0,40})?item[s]?\s*1\b", case=False, na=False)
items["usable"] = items["chars"] >= MIN_USABLE_CHARS

print(f"usable                      {int(items['usable'].sum()):>7,}  ({items['usable'].mean():.2%})")
print(f"incorporated by reference   {int(items['incorporated_by_reference'].sum()):>7,}")
print(f"no text at all              {int((items['chars'] == 0).sum()):>7,}")
print()
print(items.groupby("source")["usable"].agg(["size", "sum", "mean"]).to_string())

usable                       21,966  (98.51%)
incorporated by reference       264
no text at all                   69

          size    sum      mean
source                         
local      334    334  1.000000
none        98      0  0.000000
sec-api  21867  21632  0.989253


## Step 14. Write the table

In [9]:
COLUMNS = ["accessionNo", "cik_unpadded", "gvkey", "conm", "companyName", "formType",
           "fyear", "periodEnd", "filedAt", "filingUrl", "source", "recovery_note",
           "cut_kind", "cut_chars", "chars", "starts_at_item1",
           "incorporated_by_reference", "usable", "item1"]

out = items[COLUMNS].rename(columns={"cik_unpadded": "cik"}).sort_values(["gvkey", "fyear"])
target = PROCESSED_FOLDER / "sp500_10k_item1.parquet"
out.to_parquet(target, compression="zstd", index=False)

size = target.stat().st_size
print(f"wrote {target}")
print(f"  {len(out):,} rows, {size / 1e6:.0f} MB zstd")
print(f"  {out['item1'].str.len().sum() / 1e9:.2f} billion characters of text")
print(f"  {out['gvkey'].nunique():,} firms, fiscal years {int(out['fyear'].min())}-{int(out['fyear'].max())}")

wrote /project2/njia_1089/acroh/projects/llm_method/data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet
  22,299 rows, 145 MB zstd
  1.06 billion characters of text
  1,061 firms, fiscal years 1992-2026


## Step 15. Check the written table

Six checks on the file read back from disk. The first three stop the notebook.

- One row per S&P 500 filing.
- Accession numbers unique, and every row carries a GVKEY.
- No table or page markers and no HTML entities left in usable text.
- Usable text begins at an Item 1 heading in most rows.
- Usable rows that still hold a line-initial `Item 1A. Risk Factors` are counted.
- Coverage by decade.

In [10]:
back = pd.read_parquet(target)

assert len(back) == len(sp), f"{len(back):,} rows against {len(sp):,} S&P 500 filings"
print(f"1) one row per S&P 500 filing ({len(back):,})")

assert not back["accessionNo"].duplicated().any(), "duplicate accessionNo"
assert back["gvkey"].notna().all(), "rows without a GVKEY"
print("2) accession numbers unique, every row carries a GVKEY")

leftover = back.loc[back["usable"], "item1"].str.contains(r"##TABLE_|##\s*PAGE|<PAGE>|&#\d+;", regex=True, case=False, na=False).sum()
assert leftover == 0, f"{leftover} usable rows still carry markup artefacts"
print("3) no table/page markers or HTML entities left in usable text")

frag = back.loc[back["usable"], "item1"].str.count(r"(?<![.:;!?\)\]])\n(?=[a-z])").sum()
print(f"   mid-sentence newlines remaining: {int(frag):,}")

kept = back.loc[back["usable"], "item1"].str.contains("§", na=False).sum()
print(f"   documents still carrying a § legal citation: {int(kept):,}")

rate = back.loc[back["usable"], "starts_at_item1"].mean()
print(f"4) usable text begins at an Item 1 heading in {rate:.1%} of rows")

still = back.loc[back["usable"], "item1"].str.contains(r"(?:^|\n)[ \t]*item[ \t]*1A[ \t]*[\.\:]?[ \t]*risk", case=False, regex=True, na=False)
print(f"5) usable rows still holding a line-initial 'Item 1A. Risk Factors': {int(still.sum()):,}")

print()
print("6) coverage by decade:")
decade = back["fyear"] // 10 * 10
cov = back.groupby(decade).agg(filings=("accessionNo", "size"), usable=("usable", "sum"), median_chars=("chars", "median"))
cov.index.name = "decade"
cov["share"] = (cov["usable"] / cov["filings"]).map("{:.1%}".format)
print(cov.to_string())

1) one row per S&P 500 filing (22,299)
2) accession numbers unique, every row carries a GVKEY


3) no table/page markers or HTML entities left in usable text


   mid-sentence newlines remaining: 5,337


   documents still carrying a § legal citation: 166
4) usable text begins at an Item 1 heading in 98.0% of rows


5) usable rows still holding a line-initial 'Item 1A. Risk Factors': 0

6) coverage by decade:
        filings  usable  median_chars  share
decade                                      
1990       3987    3848       29437.0  96.5%
2000       7351    7255       39135.0  98.7%
2010       7012    6969       41372.5  99.4%
2020       3949    3894       46486.0  98.6%
